In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1Si9F7jddwRA9yFCYe4LhFXcuzEGh6Vpc", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


# Notebook 3: GPU Profiling with PyTorch Profiler

**Vizuara AI Pods | GPU Programming Course | Pod 4: Ring-AllReduce, Batch Size, and Profiling**

---

In this notebook, we will learn how to profile PyTorch training loops to identify performance bottlenecks. You will learn:

1. How to use `torch.profiler` to capture detailed timing of every operation
2. How to read profiler output and identify compute-bound, memory-bound, and communication-bound scenarios
3. How to use `record_function` to annotate your code for clear profiler traces
4. How to analyze GPU utilization and find optimization opportunities
5. Common bottlenecks and how to fix them

**Prerequisites:** Notebooks 1-2, basic PyTorch knowledge.

**Estimated time:** 40-50 minutes

**Runtime:** T4 GPU required: Runtime > Change runtime type > T4 GPU

In [ ]:
#@title 🎧 Listen: Setup Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Setup

In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
!pip install -q torch torchvision numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.profiler import profile, record_function, ProfilerActivity
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', "This notebook requires a GPU!"
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
#@title 🎧 Transition: Part1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_part1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 1: Your First Profiler Run

PyTorch's built-in profiler captures detailed timing for every CPU and CUDA operation. Let's start with a simple example to understand the basics.

In [ ]:
#@title 🎧 Code Walkthrough: Simplenet Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_simplenet_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# A simple model for profiling
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.layers(x)


model = SimpleNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Generate synthetic data
X = torch.randn(1000, 784, device=device)
y = torch.randint(0, 10, (1000,), device=device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Model created, ready to profile.")

In [ ]:
#@title 🎧 Code Walkthrough: Basic Profiling Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_basic_profiling_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Basic profiling: profile a few training steps
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step in range(5):
        with record_function("forward"):
            outputs = model(X[:128])
            loss = criterion(outputs, y[:128])

        with record_function("backward"):
            optimizer.zero_grad()
            loss.backward()

        with record_function("optimizer_step"):
            optimizer.step()

# Print the profiler results
print("\nTop 15 Operations by CUDA Time:")
print("=" * 90)
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

In [ ]:
#@title 🎧 Listen: Profiler Output Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_profiler_output_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Reading the Profiler Output

The table above shows:
- **Name:** The operation (CUDA kernel, PyTorch function, or your custom annotation)
- **Self CPU/CUDA:** Time spent in this operation only (not including sub-operations)
- **CPU/CUDA total:** Total time including sub-operations
- **Calls:** Number of times this operation was invoked

Key things to notice:
1. The **forward**, **backward**, and **optimizer_step** annotations let you see the high-level breakdown
2. Individual CUDA kernels like `aten::mm` (matrix multiply) and `aten::addmm` show where GPU time is actually spent
3. CPU time includes kernel launch overhead and Python execution

In [ ]:
#@title 🎧 Code Walkthrough: More Profiler Output Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_more_profiler_output_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Let's look at the breakdown by our custom annotations
print("\nBreakdown by Training Phase:")
print("=" * 90)
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

print("\n\nTop operations by CPU time (grouped by input shapes):")
print("=" * 90)
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cpu_time_total", row_limit=10))

In [ ]:
#@title 🎧 Transition: Part2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_part2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 2: Profiling a Real CNN Training Loop

Let's profile a more realistic scenario -- training a CNN on CIFAR-10 with proper data loading.

In [ ]:
#@title 🎧 Code Walkthrough: Cnn Data Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_cnn_data_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class CNN(nn.Module):
    """A realistic CNN for profiling."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# Load CIFAR-10
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

print(f"Dataset size: {len(train_dataset)}")
print("Ready to profile CNN training.")

In [ ]:
#@title 🎧 Code Walkthrough: Profile Training Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_profile_training_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def profile_training(batch_size, num_workers, pin_memory, num_steps=20, label=""):
    """
    Profile a training loop and return detailed timing information.
    """
    torch.manual_seed(42)
    model = CNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory, drop_last=True
    )

    model.train()

    # Use the schedule: wait 2 steps, warmup 3 steps, profile 10 steps
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=2, warmup=3, active=10, repeat=1),
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    ) as prof:
        for step, (inputs, targets) in enumerate(loader):
            if step >= num_steps:
                break

            with record_function("data_transfer"):
                inputs = inputs.to(device, non_blocking=pin_memory)
                targets = targets.to(device, non_blocking=pin_memory)

            with record_function("forward_pass"):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            with record_function("backward_pass"):
                optimizer.zero_grad()
                loss.backward()

            with record_function("optimizer_step"):
                optimizer.step()

            prof.step()

    del model, optimizer
    torch.cuda.empty_cache()

    return prof


# Profile with default settings
print("Profiling CNN training (batch_size=128, num_workers=2, pin_memory=True)...")
prof = profile_training(batch_size=128, num_workers=2, pin_memory=True)

print("\nTop 20 Operations by CUDA Time:")
print("=" * 100)
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

In [ ]:
#@title 🎧 Transition: Part3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_part3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 3: Analyzing the Training Phase Breakdown

Let's create a more intuitive visualization of where time is spent.

In [ ]:
#@title 🎧 Code Walkthrough: Extract Phase Times Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_extract_phase_times_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def extract_phase_times(prof):
    """
    Extract timing for each training phase from profiler results.
    """
    phases = {}
    for event in prof.key_averages():
        if event.key in ['data_transfer', 'forward_pass', 'backward_pass', 'optimizer_step']:
            phases[event.key] = {
                'cpu_time_ms': event.cpu_time_total / 1000,  # us to ms
                'cuda_time_ms': event.cuda_time_total / 1000,
                'calls': event.count,
                'cpu_memory_mb': event.cpu_memory_usage / 1e6 if event.cpu_memory_usage else 0,
                'cuda_memory_mb': event.cuda_memory_usage / 1e6 if event.cuda_memory_usage else 0,
            }
    return phases


phases = extract_phase_times(prof)

print("Training Phase Breakdown:")
print("=" * 70)
print(f"{'Phase':<20} {'CPU Time (ms)':>15} {'CUDA Time (ms)':>15} {'Calls':>8}")
print("-" * 70)

total_cuda = sum(p['cuda_time_ms'] for p in phases.values())

for name, data in phases.items():
    pct = data['cuda_time_ms'] / total_cuda * 100 if total_cuda > 0 else 0
    print(f"{name:<20} {data['cpu_time_ms']:>13.2f} {data['cuda_time_ms']:>13.2f} {data['calls']:>8}  ({pct:.1f}%)")

print(f"\nTotal CUDA time: {total_cuda:.2f} ms")

In [ ]:
#@title 🎧 What to Look For: Visualize Phases
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_visualize_phases.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the phase breakdown
phase_names = list(phases.keys())
cuda_times = [phases[p]['cuda_time_ms'] for p in phase_names]
cpu_times = [phases[p]['cpu_time_ms'] for p in phase_names]

# Rename for display
display_names = {
    'data_transfer': 'Data Transfer\n(CPU->GPU)',
    'forward_pass': 'Forward Pass',
    'backward_pass': 'Backward Pass',
    'optimizer_step': 'Optimizer Step',
}
labels = [display_names.get(n, n) for n in phase_names]
colors_pie = ['#90CAF9', '#4CAF50', '#FF7043', '#AB47BC']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart of CUDA time
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    cuda_times, labels=labels, autopct='%1.1f%%',
    colors=colors_pie, startangle=90,
    textprops={'fontsize': 10}
)
for autotext in autotexts:
    autotext.set_fontweight('bold')
ax.set_title('GPU Time Distribution', fontsize=14, fontweight='bold')

# Bar chart comparing CPU vs CUDA time
ax = axes[1]
x = np.arange(len(phase_names))
width = 0.35
bars1 = ax.bar(x - width/2, cpu_times, width, label='CPU Time', color='#1E88E5', alpha=0.8)
bars2 = ax.bar(x + width/2, cuda_times, width, label='CUDA Time', color='#43A047', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('CPU vs CUDA Time by Phase', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("The backward pass typically takes 2-3x the forward pass time.")
print("Data transfer should be a small fraction -- if it is large, optimize your DataLoader.")

In [ ]:
#@title 🎧 Transition: Part4 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_part4_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 4: Profiling Different Configurations

Let's compare how different DataLoader settings affect performance. This is one of the most common bottlenecks.

In [ ]:
#@title 🎧 Code Walkthrough: Compare Dataloader Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_compare_dataloader_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Compare DataLoader configurations
configs = [
    {'batch_size': 128, 'num_workers': 0, 'pin_memory': False, 'label': 'workers=0, no pin'},
    {'batch_size': 128, 'num_workers': 2, 'pin_memory': False, 'label': 'workers=2, no pin'},
    {'batch_size': 128, 'num_workers': 2, 'pin_memory': True,  'label': 'workers=2, pin=True'},
    {'batch_size': 128, 'num_workers': 4, 'pin_memory': True,  'label': 'workers=4, pin=True'},
]

config_results = []

for cfg in configs:
    print(f"\nProfiling: {cfg['label']}...")
    p = profile_training(
        batch_size=cfg['batch_size'],
        num_workers=cfg['num_workers'],
        pin_memory=cfg['pin_memory'],
        num_steps=20
    )
    phases = extract_phase_times(p)

    total_cuda = sum(v['cuda_time_ms'] for v in phases.values())
    data_pct = phases.get('data_transfer', {}).get('cuda_time_ms', 0) / total_cuda * 100 if total_cuda > 0 else 0

    config_results.append({
        'label': cfg['label'],
        'phases': phases,
        'total_cuda_ms': total_cuda,
        'data_transfer_pct': data_pct,
    })

    print(f"  Total CUDA time: {total_cuda:.1f}ms, Data transfer: {data_pct:.1f}%")

print("\nAll configurations profiled!")

In [ ]:
#@title 🎧 What to Look For: Visualize Configs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_visualize_configs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [r['label'] for r in config_results]
total_times = [r['total_cuda_ms'] for r in config_results]
data_pcts = [r['data_transfer_pct'] for r in config_results]

# Plot 1: Total CUDA time
ax = axes[0]
bar_colors = ['#E53935', '#FB8C00', '#43A047', '#1E88E5']
bars = ax.bar(range(len(labels)), total_times, color=bar_colors, alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9, rotation=15)
ax.set_ylabel('Total CUDA Time (ms)', fontsize=12)
ax.set_title('Total Training Time by Config', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, t in zip(bars, total_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{t:.1f}ms', ha='center', fontsize=10, fontweight='bold')

# Plot 2: Stacked bar showing phase breakdown
ax = axes[1]
phase_order = ['data_transfer', 'forward_pass', 'backward_pass', 'optimizer_step']
phase_colors = {'data_transfer': '#90CAF9', 'forward_pass': '#4CAF50',
                'backward_pass': '#FF7043', 'optimizer_step': '#AB47BC'}
phase_labels = {'data_transfer': 'Data Transfer', 'forward_pass': 'Forward',
                'backward_pass': 'Backward', 'optimizer_step': 'Optimizer'}

bottoms = np.zeros(len(config_results))
for phase in phase_order:
    values = [r['phases'].get(phase, {}).get('cuda_time_ms', 0) for r in config_results]
    ax.bar(range(len(labels)), values, bottom=bottoms,
           color=phase_colors[phase], label=phase_labels[phase], alpha=0.8, edgecolor='white')
    bottoms += values

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9, rotation=15)
ax.set_ylabel('CUDA Time (ms)', fontsize=12)
ax.set_title('Phase Breakdown by Config', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insight: Using num_workers>0 and pin_memory=True can significantly")
print("reduce the data loading bottleneck, keeping the GPU busy with computation.")

In [ ]:
#@title 🎧 Transition: Part5 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_part5_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 5: Profiling Individual CUDA Kernels

Let's dig deeper and look at which specific CUDA kernels dominate the execution time. This tells us whether we are compute-bound or memory-bound.

In [ ]:
#@title 🎧 Code Walkthrough: Kernel Profiling Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_kernel_profiling_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Detailed kernel-level profiling
model = CNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Pre-load data onto GPU for clean profiling (no data loading noise)
X_gpu = torch.randn(128, 3, 32, 32, device=device)
y_gpu = torch.randint(0, 10, (128,), device=device)

# Warmup
for _ in range(5):
    out = model(X_gpu)
    loss = criterion(out, y_gpu)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()

# Profile
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_flops=True,  # Enable FLOPS counting
) as prof:
    for _ in range(10):
        with record_function("forward"):
            out = model(X_gpu)
            loss = criterion(out, y_gpu)

        with record_function("backward"):
            optimizer.zero_grad()
            loss.backward()

        with record_function("optimizer"):
            optimizer.step()

# Analyze kernel-level timing
print("Top 25 CUDA Kernels by Total Time:")
print("=" * 100)
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=25))

del model, optimizer
torch.cuda.empty_cache()

In [ ]:
#@title 🎧 Code Walkthrough: Categorize Ops Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_categorize_ops_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Categorize operations
categories = defaultdict(float)

for event in prof.key_averages():
    cuda_time = event.cuda_time_total / 1000  # ms
    if cuda_time <= 0:
        continue

    name = event.key.lower()

    if 'conv' in name or 'cudnn' in name or 'winograd' in name:
        categories['Convolution'] += cuda_time
    elif 'mm' in name or 'gemm' in name or 'linear' in name or 'addmm' in name:
        categories['Matrix Multiply'] += cuda_time
    elif 'batch_norm' in name or 'batchnorm' in name:
        categories['BatchNorm'] += cuda_time
    elif 'relu' in name or 'activation' in name:
        categories['Activation'] += cuda_time
    elif 'adam' in name or 'sgd' in name or 'optimizer' in name:
        categories['Optimizer'] += cuda_time
    elif 'pool' in name:
        categories['Pooling'] += cuda_time
    elif 'copy' in name or 'memcpy' in name or 'transfer' in name:
        categories['Memory Transfer'] += cuda_time
    elif 'dropout' in name:
        categories['Dropout'] += cuda_time
    else:
        categories['Other'] += cuda_time

# Sort by time
sorted_cats = sorted(categories.items(), key=lambda x: x[1], reverse=True)
total = sum(v for _, v in sorted_cats)

print("\nOperation Category Breakdown:")
print("=" * 55)
for cat, time_ms in sorted_cats:
    pct = time_ms / total * 100
    bar = '#' * int(pct / 2)
    print(f"  {cat:<20} {time_ms:>8.2f}ms ({pct:>5.1f}%) {bar}")
print(f"\n  Total: {total:.2f}ms")

In [ ]:
#@title 🎧 What to Look For: Visualize Kernels
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_visualize_kernels.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the kernel category breakdown
fig, ax = plt.subplots(figsize=(10, 6))

cats = [c for c, _ in sorted_cats]
times = [t for _, t in sorted_cats]
pcts = [t / total * 100 for t in times]

colors_bar = plt.cm.Set2(np.linspace(0, 1, len(cats)))
bars = ax.barh(range(len(cats)), times, color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(cats)))
ax.set_yticklabels(cats, fontsize=11)
ax.set_xlabel('Total CUDA Time (ms)', fontsize=12)
ax.set_title('GPU Time by Operation Category', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

for bar, t, p in zip(bars, times, pcts):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{t:.1f}ms ({p:.0f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- If Convolution + MatMul dominate: your training is COMPUTE-BOUND (good!)")
print("- If Memory Transfer dominates: your training is MEMORY-BOUND")
print("- If Other/idle dominates: look for synchronization or data loading issues")

In [ ]:
#@title 🎧 Transition: Part6 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_part6_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 6: Batch Size Impact on GPU Utilization

Batch size directly affects GPU utilization. Let's measure how throughput and kernel efficiency change with batch size.

In [ ]:
#@title 🎧 Code Walkthrough: Measure Throughput Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_measure_throughput_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def measure_throughput(batch_size, num_steps=50):
    """Measure training throughput (samples/sec) for a given batch size."""
    torch.manual_seed(42)
    model = CNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    model.train()

    # Pre-generate data on GPU
    X = torch.randn(batch_size, 3, 32, 32, device=device)
    y = torch.randint(0, 10, (batch_size,), device=device)

    # Warmup
    for _ in range(5):
        out = model(X)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()

    # Benchmark
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(num_steps):
        out = model(X)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    samples_per_sec = batch_size * num_steps / elapsed
    ms_per_step = elapsed / num_steps * 1000

    # Memory usage
    mem_allocated = torch.cuda.max_memory_allocated() / 1e9
    torch.cuda.reset_peak_memory_stats()

    del model, optimizer
    torch.cuda.empty_cache()

    return {
        'batch_size': batch_size,
        'samples_per_sec': samples_per_sec,
        'ms_per_step': ms_per_step,
        'peak_memory_gb': mem_allocated
    }


batch_sizes = [8, 16, 32, 64, 128, 256, 512]

# Try to measure for each batch size (some may OOM)
throughput_results = []
for bs in batch_sizes:
    try:
        result = measure_throughput(bs)
        throughput_results.append(result)
        print(f"  bs={bs:>4}: {result['samples_per_sec']:>8.0f} samples/s, "
              f"{result['ms_per_step']:>6.1f} ms/step, "
              f"peak mem={result['peak_memory_gb']:.2f} GB")
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f"  bs={bs:>4}: OUT OF MEMORY")
            torch.cuda.empty_cache()
        else:
            raise

In [ ]:
#@title 🎧 What to Look For: Visualize Throughput
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_visualize_throughput.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize throughput results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bs_vals = [r['batch_size'] for r in throughput_results]
throughputs = [r['samples_per_sec'] for r in throughput_results]
step_times = [r['ms_per_step'] for r in throughput_results]
memories = [r['peak_memory_gb'] for r in throughput_results]

# Plot 1: Throughput
ax = axes[0]
ax.plot(bs_vals, throughputs, 'o-', color='#1E88E5', linewidth=2, markersize=8)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Throughput (samples/sec)', fontsize=12)
ax.set_title('Training Throughput', fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

# Plot 2: Time per step
ax = axes[1]
ax.plot(bs_vals, step_times, 'o-', color='#43A047', linewidth=2, markersize=8)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Time per Step (ms)', fontsize=12)
ax.set_title('Step Latency', fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

# Plot 3: Memory usage
ax = axes[2]
ax.plot(bs_vals, memories, 'o-', color='#E53935', linewidth=2, markersize=8)
total_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
ax.axhline(y=total_mem, color='red', linestyle='--', alpha=0.5, label=f'GPU limit ({total_mem:.0f} GB)')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Peak Memory (GB)', fontsize=12)
ax.set_title('GPU Memory Usage', fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Batch Size vs GPU Utilization', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("1. Throughput increases with batch size (better GPU utilization)")
print("2. Step time increases sub-linearly (GPU processes more in parallel)")
print("3. Memory grows linearly with batch size (activations stored for backward)")
print("4. There is a maximum batch size before you hit OOM")

In [ ]:
#@title 🎧 Transition: Part7 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_part7_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 7: Mixed Precision Profiling

Mixed precision (FP16/BF16) can dramatically reduce both compute time and memory usage. Let's profile with and without it.

In [ ]:
#@title 🎧 Code Walkthrough: Benchmark Amp Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_benchmark_amp_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def benchmark_precision(batch_size=128, use_amp=False, num_steps=50):
    """Benchmark training with and without automatic mixed precision."""
    torch.manual_seed(42)
    model = CNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    model.train()

    X = torch.randn(batch_size, 3, 32, 32, device=device)
    y = torch.randint(0, 10, (batch_size,), device=device)

    # Warmup
    for _ in range(5):
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(X)
            loss = criterion(out, y)
        optimizer.zero_grad()
        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()

    # Benchmark
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(num_steps):
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(X)
            loss = criterion(out, y)
        optimizer.zero_grad()
        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    result = {
        'mode': 'AMP (FP16)' if use_amp else 'FP32',
        'samples_per_sec': batch_size * num_steps / elapsed,
        'ms_per_step': elapsed / num_steps * 1000,
        'peak_memory_gb': torch.cuda.max_memory_allocated() / 1e9,
    }

    del model, optimizer
    torch.cuda.empty_cache()
    return result


print("Comparing FP32 vs Mixed Precision (AMP):")
print("=" * 60)

for bs in [64, 128, 256]:
    fp32 = benchmark_precision(bs, use_amp=False)
    amp = benchmark_precision(bs, use_amp=True)
    speedup = amp['samples_per_sec'] / fp32['samples_per_sec']
    mem_saving = (1 - amp['peak_memory_gb'] / fp32['peak_memory_gb']) * 100

    print(f"\n  Batch size = {bs}:")
    print(f"    FP32:     {fp32['samples_per_sec']:>8.0f} samples/s, {fp32['peak_memory_gb']:.2f} GB")
    print(f"    AMP:      {amp['samples_per_sec']:>8.0f} samples/s, {amp['peak_memory_gb']:.2f} GB")
    print(f"    Speedup:  {speedup:.2f}x,  Memory saved: {mem_saving:.0f}%")

In [ ]:
#@title 🎧 Transition: Part8 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_part8_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 8: Simulating Communication Bottlenecks

In distributed training, NCCL allreduce operations appear in the profiler. We can simulate this by inserting artificial communication delays to understand how they affect the training timeline.

In [ ]:
#@title 🎧 Code Walkthrough: Simulate Dist Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_simulate_dist_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def simulate_distributed_step(model, X, y, criterion, optimizer, comm_time_ms=0):
    """
    Simulate one distributed training step, including artificial
    communication time to represent allreduce overhead.
    """
    with record_function("forward"):
        out = model(X)
        loss = criterion(out, y)

    with record_function("backward"):
        optimizer.zero_grad()
        loss.backward()

    # Simulate allreduce communication
    if comm_time_ms > 0:
        with record_function("simulated_allreduce"):
            torch.cuda.synchronize()
            time.sleep(comm_time_ms / 1000)  # Simulate network wait

    with record_function("optimizer"):
        optimizer.step()

    return loss.item()


# Measure the impact of communication overhead
model = CNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
model.train()

X = torch.randn(128, 3, 32, 32, device=device)
y = torch.randint(0, 10, (128,), device=device)

# Warmup
for _ in range(5):
    simulate_distributed_step(model, X, y, criterion, optimizer, 0)

comm_times = [0, 1, 2, 5, 10, 20]  # ms of simulated communication delay
throughputs_comm = []

print("Impact of Communication Overhead on Training Throughput:")
print("=" * 60)

for comm_ms in comm_times:
    num_steps = 20
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(num_steps):
        simulate_distributed_step(model, X, y, criterion, optimizer, comm_ms)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    throughput = 128 * num_steps / elapsed
    throughputs_comm.append(throughput)
    compute_pct = (1 - comm_ms * num_steps / 1000 / elapsed) * 100

    print(f"  Comm overhead = {comm_ms:>2}ms: {throughput:>7.0f} samples/s  "
          f"(compute utilization: {compute_pct:.0f}%)")

del model, optimizer
torch.cuda.empty_cache()

In [ ]:
#@title 🎧 What to Look For: Visualize Comm
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_visualize_comm.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize communication overhead impact
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(comm_times, throughputs_comm, 'o-', color='#1E88E5', linewidth=2, markersize=8)
ax.axhline(y=throughputs_comm[0], color='gray', linestyle='--', alpha=0.5,
           label=f'No comm overhead: {throughputs_comm[0]:.0f} samples/s')

ax.set_xlabel('Simulated Communication Delay (ms)', fontsize=13)
ax.set_ylabel('Throughput (samples/sec)', fontsize=13)
ax.set_title('Impact of Communication Overhead on Training Throughput\n'
             '(Simulated AllReduce Delay)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate the key insight
if len(throughputs_comm) > 3:
    ax.annotate(f'{throughputs_comm[-1]:.0f} samples/s\n({throughputs_comm[-1]/throughputs_comm[0]*100:.0f}% of baseline)',
                xy=(comm_times[-1], throughputs_comm[-1]),
                xytext=(comm_times[-1] - 5, throughputs_comm[-1] + 200),
                fontsize=11, fontweight='bold', color='#E53935',
                arrowprops=dict(arrowstyle='->', color='#E53935'))

plt.tight_layout()
plt.show()

print("\nThis is why communication-computation overlap is critical:")
print("Without overlap, even a few ms of allreduce latency per step")
print("can significantly reduce GPU utilization.")

## Exercises


In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Profile Forward vs Backward by Layer

Use `record_function` to annotate each block of the CNN (conv block 1, 2, 3, and the classifier). Profile the model and determine which layers take the most time in the forward pass vs backward pass.

In [ ]:
#@title 🎧 Before You Start: Todo1 Hints
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_30_todo1_hints.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 1: Profile per-layer timing
#
# Create a modified CNN that uses record_function to annotate each block.
# For example:
#
# class ProfiledCNN(nn.Module):
#     def forward(self, x):
#         with record_function("conv_block_1"):
#             x = self.block1(x)
#         with record_function("conv_block_2"):
#             x = self.block2(x)
#         ...
#
# Then profile and extract timing for each block.
# Which block takes the most time? Is the time proportional to
# the number of parameters?

print("TODO: Create a ProfiledCNN and measure per-layer timing.")
print("Hint: Use record_function context managers in the forward method.")
print("The backward pass will automatically be associated with the same scopes.")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_31_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Optimize a Bottlenecked Training Loop

Below is a deliberately inefficient training loop. Use the profiler to identify bottlenecks and fix them. Your goal is to at least double the throughput.

In [ ]:
#@title 🎧 Before You Start: Todo2 Hints
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_32_todo2_hints.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Find and fix the bottlenecks
#
# This training loop has AT LEAST 4 performance problems.
# Profile it, identify the issues, and fix them.

def slow_training_loop(num_steps=50):
    model = CNN().to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    model.train()

    # Problem 1: num_workers=0 (data loading on main thread)
    loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                        num_workers=0, pin_memory=False)

    start = time.time()
    for step, (inputs, targets) in enumerate(loader):
        if step >= num_steps:
            break

        # Problem 2: synchronous transfer without pin_memory
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Problem 3: printing loss every step forces CPU-GPU sync
        print(f"Step {step}: loss = {loss.item():.4f}", end='\r')

        # Problem 4: unnecessary GPU sync every step
        torch.cuda.synchronize()

    elapsed = time.time() - start
    throughput = 32 * num_steps / elapsed
    print(f"\nSlow loop: {throughput:.0f} samples/s ({elapsed:.1f}s for {num_steps} steps)")

    del model, optimizer
    torch.cuda.empty_cache()
    return throughput


print("Running SLOW training loop...")
slow_tp = slow_training_loop()

# TODO: Write an optimized version that fixes all 4 problems
# def fast_training_loop(num_steps=50):
#     ...

print(f"\nTODO: Implement fast_training_loop() and achieve > {slow_tp * 2:.0f} samples/s")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_33_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 3: Memory Profiling Deep Dive

Use `torch.cuda.memory_stats()` and the profiler's memory tracking to analyze memory usage during a training step. Create a plot showing memory usage over time within a single training step (before forward, after forward, after backward, after optimizer step). How much memory is used by activations vs gradients vs optimizer state?

In [ ]:
#@title 🎧 Before You Start: Todo3 Hints
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_34_todo3_hints.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 3: Memory profiling
#
# Steps:
# 1. Record memory usage at different points in the training step
# 2. Calculate the memory used by:
#    - Model parameters (constant)
#    - Activations (forward pass, freed during backward)
#    - Gradients (created during backward)
#    - Optimizer states (e.g., Adam's m and v buffers)
# 3. Plot memory usage over time within one step
#
# Useful functions:
#   torch.cuda.memory_allocated()
#   torch.cuda.max_memory_allocated()
#   torch.cuda.memory_reserved()
#   torch.cuda.reset_peak_memory_stats()

print("TODO: Implement memory profiling for a single training step")
print("Expected memory breakdown (rough estimates):")
print("  Parameters:       ~10 MB  (2.4M params x 4 bytes)")
print("  Gradients:        ~10 MB  (same size as parameters)")
print("  Optimizer states: ~20 MB  (Adam stores m and v, each param-sized)")
print("  Activations:      variable (depends on batch size)")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_35_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, we have:

1. **Learned to use `torch.profiler`** with `record_function` annotations for clear, structured profiler output

2. **Analyzed training phase breakdown** -- forward, backward, optimizer, and data transfer -- to identify where time is spent

3. **Compared DataLoader configurations** and seen how `num_workers` and `pin_memory` affect data loading bottlenecks

4. **Profiled individual CUDA kernels** to determine whether training is compute-bound, memory-bound, or communication-bound

5. **Measured batch size effects** on throughput, step time, and GPU memory usage

6. **Compared FP32 vs mixed precision** and seen the speedup and memory savings from AMP

7. **Simulated communication overhead** to understand how allreduce latency affects distributed training throughput

### Key Takeaways

- Always profile before optimizing -- do not guess where the bottleneck is
- Use `record_function` liberally to make traces readable
- Common bottlenecks: data loading (fix with workers/pin_memory), small batch sizes (increase to saturate GPU), unnecessary synchronizations (avoid .item() in hot loops)
- Mixed precision is almost always a win: faster compute + less memory
- In distributed training, communication overhead is visible as NCCL operations in the profiler
- Target >80% GPU utilization; world-class pipelines achieve 50-60% MFU

### Putting It All Together

With the knowledge from all three notebooks, you can now:
1. Understand how GPUs communicate (ring-allreduce)
2. Choose the right batch size for distributed training
3. Profile and optimize your training loops

These are the foundational skills for efficient GPU programming in deep learning.